In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [14]:
df = pd.read_csv("train_clean.csv")

In [15]:
df = df.drop('id', axis=1)
X = df.drop("Will_Buy_EV", axis=1)
y = df["Will_Buy_EV"]

num_col = []
cat_col = []

for col in X.columns:
    if pd.api.types.is_numeric_dtype(X[col]):
        num_col.append(col)
    else:
        cat_col.append(col)

for col in cat_col:
    X[col] = X[col].astype('category')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [16]:
import optuna
from catboost import CatBoostClassifier

# 1. Split data training khusus untuk validasi Optuna (Holdout)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

def objective(trial):
    # 2. Search space parameter krusial untuk CatBoost
    param = {
        'iterations': 1500, # Set tinggi, dikontrol early stopping
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'depth': trial.suggest_int('depth', 4, 10), # Max 10 agar tidak terlalu lambat
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
        'bootstrap_type': 'Bernoulli', # Syarat mutlak agar parameter subsample jalan
        'random_seed': 42,
        'eval_metric': 'AUC',
        'verbose': 3,
        'auto_class_weights': 'Balanced'
    }

    model = CatBoostClassifier(**param)

    # 3. Fit dengan Early Stopping dan penentuan cat_features
    model.fit(
        X_tr, y_tr,
        cat_features=cat_col, # Parameter WAJIB CatBoost untuk data kategorikal
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=30,
        verbose=False # Matikan spam output iterasi
    )

    # 4. Evaluasi ROC-AUC
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_pred_proba)

# 5. Eksekusi Optuna
study = optuna.create_study(
    direction='maximize', 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10)
)
study.optimize(objective, n_trials=30) 


[I 2026-09-13 18:34:15,123] A new study created in memory with name: no-name-c0eb3454-75cc-4147-bb6a-e23f3fb185aa
[I 2026-09-13 18:48:11,642] Trial 0 finished with value: 0.9406330194089618 and parameters: {'learning_rate': 0.028976973850469233, 'depth': 9, 'l2_leaf_reg': 1.8314594502408563, 'subsample': 0.7753023758585889, 'colsample_bylevel': 0.9535058637711719}. Best is trial 0 with value: 0.9406330194089618.
[I 2026-09-13 18:52:49,861] Trial 1 finished with value: 0.9403052459052222 and parameters: {'learning_rate': 0.06114823083612469, 'depth': 10, 'l2_leaf_reg': 7.492810519988512, 'subsample': 0.7344972124246288, 'colsample_bylevel': 0.6329902934975663}. Best is trial 0 with value: 0.9406330194089618.
[I 2026-09-13 18:57:17,397] Trial 2 finished with value: 0.9409729680272872 and parameters: {'learning_rate': 0.06931997626973475, 'depth': 6, 'l2_leaf_reg': 2.494094464334164, 'subsample': 0.8444010814185028, 'colsample_bylevel': 0.6119961203689824}. Best is trial 2 with value: 0.9

In [17]:
print("Parameter Terbaik:", study.best_params)
print("ROC-AUC Tertinggi di Optuna:", study.best_value)

Parameter Terbaik: {'learning_rate': 0.17744619808901926, 'depth': 4, 'l2_leaf_reg': 4.047995699133564, 'subsample': 0.9861496453487024, 'colsample_bylevel': 0.8388400997630279}
ROC-AUC Tertinggi di Optuna: 0.9413558798660692
